In [21]:
import time
import pandas as pd
import json
import os
from google import genai

# ۱. خواندن کلید جمنای از فایل config.json
try:
    with open("config.json", "r", encoding="utf-8") as f:
        config = json.load(f)
        GEMINI_API_KEY = config.get("GEMINI_KEY")
except Exception as e:
    print(f"❌ خطا در خواندن فایل config.json: {e}")
    exit()

# راه‌اندازی کلاینت جمنای با مدل gemini-3.8-flash
client = genai.Client(api_key=GEMINI_API_KEY)

# تمرکز فقط روی بلوار پاسداران شیراز
TARGET_ZONE = "منطقه ۹ / منطقه ۴"
TARGET_STREET = "بلوار پاسداران (مقاومت)"

# جامع‌ترین دسته‌بندی اصناف برای پوشش ۱۰۰٪ تمام کسب‌وکارها
ALL_CATEGORIES = [
    "رستوران، فست‌فود، کافه و قنادی",
    "سوپرمارکت، هایپرمارکت و مواد غذایی",
    "پوشاک، کیف، کفش و طلا",
    "لوازم خانگی، صوتی و تصویری و دیجیتال",
    "خدمات خودرویی، تعمیرگاه، لوازم یدکی و کارواش",
    "داروخانه، مطب، کلینیک و تجهیزات پزشکی",
    "املاک، ساخت‌وساز و دکوراسیون داخلی",
    "خدمات آرایشی، زیبایی و سالن‌های پیرایشی",
    "مصالح ساختمانی، ابزارآلات و تأسیسات",
    "خدمات آموزشی، آموزشگاه‌ها و مهدکودک",
    "بانک، بیمه، خدمات مالی و دفتری",
    "مجارری تجاری، پاساژها و سایر مشاغل خدماتی"
]

OUTPUT_CSV = "shiraz_pasdaran_all_businesses.csv"
OUTPUT_EXCEL = "shiraz_pasdaran_all_businesses.xlsx"

def generate_all_businesses_pasdaran(street_name, category_name):
    """استخراج حداکثری و جامع تمام اصناف یک صنف در بلوار پاسداران"""
    prompt = f"""
    شما یک متخصص بازاریابی و پایگاه داده شهری در شیراز هستید.
    لطفاً لیست کامل، حداکثری و تمام‌عیار از تمامی کسب‌وکارها، فروشگاه‌ها، شرکت‌ها و مراکز واقعی و فعالی که در شاخه یا صنف '{category_name}' واقع در '{street_name}' در شیراز فعال هستند را استخراج کنید.
    هیچ‌گونه محدودیتی در تعداد وجود ندارد؛ تا جایی که می‌توانید مراکز واقعی، نام‌ها، مغازه‌ها و مؤسسات این محدوده را به صورت دقیق و کامل لیست کنید.

    خروجی را دقیقاً و فقط به صورت یک آرایه JSON معتبر (بدون هیچ متن اضافی یا توضیحات خارج از JSON) با کلیدهای زیر برگردانید:
    [
      {{
        "main_street": "{street_name}",
        "store_name": "نام دقیق فروشگاه، شرکت یا مرکز",
        "address": "آدرس دقیق، پلاک یا محدوده دقیق در بلوار پاسداران",
        "category": "{category_name}",
        "phone_fixed": "شماره تلفن ثابت با پیش‌شماره 071 (اگر موجود بود)",
        "phone_mobile": "شماره موبایل (اگر موجود بود)"
      }}
    ]
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model='gemini-3.8-flash',
                contents=prompt
            )
            text = response.text.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.endswith("```"):
                text = text[:-3]
            return json.loads(text.strip())
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"  ⚠️ شلوغی موقت سرور جمنای (تلاش {attempt + 1} از ۳)... ۵ ثانیه صبر می‌کنیم.")
                time.sleep(5)
            else:
                print(f"خطا در پردازش هوش مصنوعی: {e}")
                break
    return []

def save_batch_to_file(batch_data):
    """ذخیره لحظه‌ای روی فایل CSV (Checkpointing)"""
    if not batch_data:
        return
    df_batch = pd.DataFrame(batch_data)
    file_exists = os.path.isfile(OUTPUT_CSV)
    df_batch.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')

def main():
    total_saved = 0
    print(f"🚀 شروع استخراج جامع تمامی اصناف در '{TARGET_STREET}' شیراز...")

    for cat in ALL_CATEGORIES:
        print(f"در حال استخراج دسته: '{cat}'...")
        
        raw_items = generate_all_businesses_pasdaran(TARGET_STREET, cat)
        
        batch_results = []
        if isinstance(raw_items, list):
            for item in raw_items:
                batch_results.append({
                    "محدوده جغرافیایی": TARGET_ZONE,
                    "خیابان اصلی": item.get("main_street", TARGET_STREET),
                    "نام فروشگاه": item.get("store_name", ""),
                    "آدرس": item.get("address", f"شیراز، {TARGET_STREET}"),
                    "نوع کاربری": item.get("category", cat),
                    "شماره تماس ثابت": item.get("phone_fixed", ""),
                    "شماره تماس موبایل": item.get("phone_mobile", "")
                })

        if batch_results:
            save_batch_to_file(batch_results)
            total_saved += len(batch_results)
            print(f"  └─ 💾 تعداد {len(batch_results)} رکورد استخراج و ذخیره شد (مجموع: {total_saved})")

        # مکث کوتاه بین دسته‌ها
        time.sleep(1.5)

    # تبدیل نهایی به اکسل و حذف موارد تکراری احتمالی
    if os.path.exists(OUTPUT_CSV):
        df_final = pd.read_csv(OUTPUT_CSV)
        df_final.drop_duplicates(subset=["نام فروشگاه", "آدرس"], inplace=True)
        df_final.to_excel(OUTPUT_EXCEL, index=False)
        print(f"\n✅ عملیات استخراج بلوار پاسداران با موفقیت کامل انجام شد!")
        print(f"📊 کل رکوردهای یکتا استخراج‌شده: {len(df_final)}")
        print(f"📁 فایل نهایی اکسل: {OUTPUT_EXCEL}")
    else:
        print("❌ داده‌ای ذخیره نشد.")

if __name__ == "__main__":
    main()

🚀 شروع استخراج جامع تمامی اصناف در 'بلوار پاسداران (مقاومت)' شیراز...
در حال استخراج دسته: 'رستوران، فست‌فود، کافه و قنادی'...
  └─ 💾 تعداد 30 رکورد استخراج و ذخیره شد (مجموع: 30)
در حال استخراج دسته: 'سوپرمارکت، هایپرمارکت و مواد غذایی'...
  └─ 💾 تعداد 20 رکورد استخراج و ذخیره شد (مجموع: 50)
در حال استخراج دسته: 'پوشاک، کیف، کفش و طلا'...
  └─ 💾 تعداد 20 رکورد استخراج و ذخیره شد (مجموع: 70)
در حال استخراج دسته: 'لوازم خانگی، صوتی و تصویری و دیجیتال'...
  └─ 💾 تعداد 20 رکورد استخراج و ذخیره شد (مجموع: 90)
در حال استخراج دسته: 'خدمات خودرویی، تعمیرگاه، لوازم یدکی و کارواش'...
  └─ 💾 تعداد 18 رکورد استخراج و ذخیره شد (مجموع: 108)
در حال استخراج دسته: 'داروخانه، مطب، کلینیک و تجهیزات پزشکی'...
  └─ 💾 تعداد 20 رکورد استخراج و ذخیره شد (مجموع: 128)
در حال استخراج دسته: 'املاک، ساخت‌وساز و دکوراسیون داخلی'...
  └─ 💾 تعداد 25 رکورد استخراج و ذخیره شد (مجموع: 153)
در حال استخراج دسته: 'خدمات آرایشی، زیبایی و سالن‌های پیرایشی'...
  ⚠️ شلوغی موقت سرور جمنای (تلاش 1 از ۳)... ۵ ثانیه صبر می‌کنیم.
